# Script for splitting each of the models in the receptor folder into separate pdb files, and depositing them into a new folder

In [1]:
import os
from pathlib import Path

_pipeline_dir = Path(os.getcwd()).parent

# Change only this line to switch dataset (e.g. 'N249G_AZ_structures')
_dataset = 'raw_files'

input_root  = str(_pipeline_dir / _dataset)
output_root = str(_pipeline_dir / 'processed_files')

os.makedirs(output_root, exist_ok=True)

print(f"input_root  : {input_root}")
print(f"output_root : {output_root}")


input_root  : /home/oliverto/Degree_projectVM/final_pipeline/raw_files
output_root : /home/oliverto/Degree_projectVM/final_pipeline/processed_files


In [2]:
import os
source_folder = os.path.join(input_root, 'receptor')
target_folder = os.path.join(output_root, 'receptor_processed/receptor_0_split')

if not os.path.exists(target_folder):
    os.makedirs(target_folder)

def process_all_pdbs(source_dir, target_dir):
    for filename in os.listdir(source_dir):
        if filename.endswith(".pdb"):
            input_path = os.path.join(source_dir, filename)
            base_name = os.path.splitext(filename)[0]
            
            print(f"Processing: {filename}")
            
            with open(input_path, 'r') as f:
                lines = f.readlines()
            
            model_num = 1
            current_model = []
            
            for line in lines:
                # Detect the start of a new model
                if line.startswith('MODEL'):
                    # If we were already tracking a model but didn't hit ENDMDL yet
                    if current_model:
                        save_model(current_model, target_dir, base_name, model_num)
                        model_num += 1
                    current_model = [line]
                
                elif line.startswith('ENDMDL'):
                    current_model.append(line)
                    save_model(current_model, target_dir, base_name, model_num)
                    model_num += 1
                    current_model = []
                
                # Ignore global END/MASTER tags while inside a model block
                elif line.startswith('END') and not line.startswith('ENDMDL'):
                    continue
                
                else:
                    current_model.append(line)
            
            # Catch the last model if the file didn't end with ENDMDL
            if current_model and len(current_model) > 1:
                save_model(current_model, target_dir, base_name, model_num)

def save_model(model_lines, output_dir, base_name, num):
    """Helper to write the model file and ensure it ends cleanly."""
    output_file = os.path.join(output_dir, f"{base_name}_lig{num}.pdb")
    
    # Ensure there's an END tag at the very bottom of every split file
    if not model_lines[-1].strip() == "END":
        model_lines.append("END\n")
        
    with open(output_file, 'w') as out:
        out.writelines(model_lines)
    print(f"   -> Saved {base_name}_lig{num}.pdb")

process_all_pdbs(source_folder, target_folder)
print("\nAll PDB models have been split.")


Processing: AEUYGRJBDIISBC-MXQXRXRHNA-O_out.pdb
   -> Saved AEUYGRJBDIISBC-MXQXRXRHNA-O_out_lig1.pdb
   -> Saved AEUYGRJBDIISBC-MXQXRXRHNA-O_out_lig2.pdb
   -> Saved AEUYGRJBDIISBC-MXQXRXRHNA-O_out_lig3.pdb
   -> Saved AEUYGRJBDIISBC-MXQXRXRHNA-O_out_lig4.pdb
Processing: APPWDWVMENBZFH-UHFFFAOYNA-N_out.pdb
   -> Saved APPWDWVMENBZFH-UHFFFAOYNA-N_out_lig1.pdb
   -> Saved APPWDWVMENBZFH-UHFFFAOYNA-N_out_lig2.pdb
   -> Saved APPWDWVMENBZFH-UHFFFAOYNA-N_out_lig3.pdb

All PDB models have been split.


# Fix pdb file by adding LIG label, correcting element type 

In [3]:
import os
source_folder = os.path.join(output_root, 'receptor_processed/receptor_0_split')
target_folder = os.path.join(output_root, 'receptor_processed/receptor_1_label')

if not os.path.exists(target_folder):
    os.makedirs(target_folder)

def fix_ligand_pdb_fixed_width(input_path, output_path):
    with open(input_path, 'r') as f:
        lines = f.readlines()

    ligand_atom_no_ticker = 1
    new_lines = []

    for line in lines:
        if line.startswith('ATOM') or line.startswith('HETATM'):
            # Using exact PDB column slicing
            record  = line[0:6]    # ATOM/HETATM
            serial  = line[6:11]   # Atom ID
            name    = line[12:16]  # Atom Name
            resName = line[17:20]  # Residue Name
            chainID = line[21:22]  # Chain
            resSeq  = line[22:26]  # Residue Index
            coords  = line[30:54]  # X, Y, Z
            occ     = line[54:60]  # Occupancy
            temp    = line[60:66]  # Temp factor
            element_col = line[76:78].strip()

            # element logic
            if element_col == 'A':
                element = 'C'
            elif len(element_col) > 1:
                element = element_col[0].upper()
            elif not element_col: # If element col is empty, guess from name
                element = name.strip()[0]
            else:
                element = element_col

            # UNL to LIG
            if resName.strip() == 'UNL':
                resName = 'LIG'
                chainID = 'B'
                resSeq  = f"{ligand_atom_no_ticker:>4}"
                ligand_atom_no_ticker += 1

            formatted_line = (
                f"{record}{serial} {name} {resName} {chainID}{resSeq}    "
                f"{coords}{occ}{temp}          {element:>2s}\n"
            )
            new_lines.append(formatted_line)
        else:
            new_lines.append(line)

    with open(output_path, 'w') as f:
        f.writelines(new_lines)

# Process the directory
for filename in os.listdir(source_folder):
    if filename.endswith(".pdb"):
        base_name = os.path.splitext(filename)[0]
        output_filename = f"{base_name}.pdb"
        fix_ligand_pdb_fixed_width(os.path.join(source_folder, filename), 
                                   os.path.join(target_folder, output_filename))
        print(f"Fixed formatting for: {output_filename}")


Fixed formatting for: AEUYGRJBDIISBC-MXQXRXRHNA-O_out_lig1.pdb
Fixed formatting for: AEUYGRJBDIISBC-MXQXRXRHNA-O_out_lig2.pdb
Fixed formatting for: AEUYGRJBDIISBC-MXQXRXRHNA-O_out_lig3.pdb
Fixed formatting for: AEUYGRJBDIISBC-MXQXRXRHNA-O_out_lig4.pdb
Fixed formatting for: APPWDWVMENBZFH-UHFFFAOYNA-N_out_lig1.pdb
Fixed formatting for: APPWDWVMENBZFH-UHFFFAOYNA-N_out_lig2.pdb
Fixed formatting for: APPWDWVMENBZFH-UHFFFAOYNA-N_out_lig3.pdb


# Code for placing all atoms in correct residue-blocks

In [4]:
import MDAnalysis as mda
import numpy as np

input_dir = os.path.join(output_root, 'receptor_processed/receptor_1_label')
output_dir = os.path.join(output_root, 'receptor_processed/receptor_2_atomblocks')

def fix_residue_blocks(src, dest):
    if not os.path.exists(dest):
        os.makedirs(dest)

    for filename in os.listdir(src):
        if filename.endswith(".pdb"):
            input_path = os.path.join(src, filename)
            output_path = os.path.join(dest, filename)
            
            u = mda.Universe(input_path)

            # clear segment IDs
            for seg in u.segments:
                seg.segid = ''

            # sort atoms into contiguous residue blocks
            selection = u.atoms
            sorted_indices = np.lexsort((selection.resnums, selection.segids))
            sorted_atoms = selection[sorted_indices]

            # re-index serials 1 to N
            sorted_atoms.ids = np.arange(1, len(sorted_atoms) + 1)

            with mda.Writer(output_path, sorted_atoms.n_atoms) as W:
                for ts in u.trajectory:
                    W.write(sorted_atoms)
            
            print(f"Atoms re-indexed to correct atom-block: {filename}")

fix_residue_blocks(input_dir, output_dir)


/home/oliverto/envs/gromacs-env/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/home/oliverto/envs/gromacs-env/lib/python3.14/site-packages/MDAnalysis/coordinates/PDB.py:885: UserWarning: Unit cell dimensions not found. CRYST1 record set to unitary values.
  warnings.warn(
/home/oliverto/envs/gromacs-env/lib/python3.14/site-packages/MDAnalysis/coordinates/PDB.py:1282: UserWarning: Found no information for attr: 'formalcharges' Using default value of '0'
  warnings.warn(


Atoms re-indexed to correct atom-block: AEUYGRJBDIISBC-MXQXRXRHNA-O_out_lig1.pdb
Atoms re-indexed to correct atom-block: AEUYGRJBDIISBC-MXQXRXRHNA-O_out_lig2.pdb
Atoms re-indexed to correct atom-block: AEUYGRJBDIISBC-MXQXRXRHNA-O_out_lig3.pdb
Atoms re-indexed to correct atom-block: AEUYGRJBDIISBC-MXQXRXRHNA-O_out_lig4.pdb
Atoms re-indexed to correct atom-block: APPWDWVMENBZFH-UHFFFAOYNA-N_out_lig1.pdb
Atoms re-indexed to correct atom-block: APPWDWVMENBZFH-UHFFFAOYNA-N_out_lig2.pdb
Atoms re-indexed to correct atom-block: APPWDWVMENBZFH-UHFFFAOYNA-N_out_lig3.pdb


# script that slices the ligand and cofactor out of the pdb-file while correcting atom-numbering ahead fo hydrogen addition through GROMACS 

In [5]:

input_dir = os.path.join(output_root, 'receptor_processed/receptor_2_atomblocks')
output_dir = os.path.join(output_root, 'receptor_processed/receptor_3_onlyprotein')

def slice_pdb_folder(source_folder, target_folder, residues_to_remove):
    """
    Processes all PDB files in a folder, removing specific residues 
    and re-indexing atom serial numbers.
    """
    if not os.path.exists(target_folder):
        os.makedirs(target_folder)

    # Convert residues to a set for faster lookup
    to_remove = set(residues_to_remove)

    for filename in os.listdir(source_folder):
        if filename.endswith(".pdb"):
            input_path = os.path.join(source_folder, filename)
            output_path = os.path.join(target_folder, filename)
            
            new_lines = []
            atom_count = 1
            
            with open(input_path, 'r') as f:
                for line in f:
                    if line.startswith("ATOM  ") or line.startswith("HETATM"):
                        # PDB standard: Residue name is strictly columns 17-20
                        res_name = line[17:20].strip()
                        
                        if res_name not in to_remove:
                            serial_str = f"{atom_count:>5}"
                            updated_line = line[:6] + serial_str + line[11:]
                            new_lines.append(updated_line)
                            atom_count += 1
                    
                    elif line.startswith("TER"):
                        new_lines.append("TER\n")
                    elif line.startswith("END"):
                        new_lines.append("END\n")

            with open(output_path, 'w') as f:
                f.writelines(new_lines)
            
            print(f"Processed {filename}: Kept {atom_count - 1} atoms.")




targets = ['LIG', 'SAH'] # "residue types" to be cut

slice_pdb_folder(input_dir, output_dir, targets)


Processed AEUYGRJBDIISBC-MXQXRXRHNA-O_out_lig1.pdb: Kept 2459 atoms.
Processed AEUYGRJBDIISBC-MXQXRXRHNA-O_out_lig2.pdb: Kept 2459 atoms.
Processed AEUYGRJBDIISBC-MXQXRXRHNA-O_out_lig3.pdb: Kept 2459 atoms.
Processed AEUYGRJBDIISBC-MXQXRXRHNA-O_out_lig4.pdb: Kept 2459 atoms.
Processed APPWDWVMENBZFH-UHFFFAOYNA-N_out_lig1.pdb: Kept 2459 atoms.
Processed APPWDWVMENBZFH-UHFFFAOYNA-N_out_lig2.pdb: Kept 2459 atoms.
Processed APPWDWVMENBZFH-UHFFFAOYNA-N_out_lig3.pdb: Kept 2459 atoms.


In [6]:
# Necessary for system this pipeline was developed on
path=os.environ.get('PATH')
%env PATH=~/envs/gromacs-env/bin:$path

env: PATH=~/envs/gromacs-env/bin:/home/oliverto/envs/gromacs-env/bin:/home/oliverto/.vscode-server/cli/servers/Stable-07ff9d6178ede9a1bd12ad3399074d726ebe6e43/server/bin/remote-cli:/home/oliverto/.local/bin:/home/oliverto/bin:/usr/local/bin:/usr/bin:/usr/local/sbin:/usr/sbin:/home/oliverto/.vscode-server/cli/servers/Stable-07ff9d6178ede9a1bd12ad3399074d726ebe6e43/server/bin/remote-cli:/home/oliverto/.local/bin:/home/oliverto/bin:/usr/local/bin:/usr/bin:/usr/local/sbin:/usr/sbin


# Perform hydrogen addition and energyminimization on protein-only structures with frozen heavy atoms. 

In [7]:
import subprocess

input_dir = os.path.join(output_root, 'receptor_processed/receptor_3_onlyprotein')
base_work_dir = os.path.join(output_root, 'receptor_processed/gromacs/batch_gromacs_runs')
gmx_path = "/home/oliverto/envs/gromacs-env/bin/gmx"

os.makedirs(base_work_dir, exist_ok=True)
pdb_files = [f for f in os.listdir(input_dir) if f.endswith('.pdb')]

print(f"Found {len(pdb_files)} structures.")

for pdb in pdb_files:
    struct_id = pdb.replace(".pdb", "")
    run_dir = os.path.join(base_work_dir, struct_id)
    os.makedirs(run_dir, exist_ok=True)
    
    input_path = os.path.join(input_dir, pdb)
    processed_gro = os.path.join(run_dir, f"{struct_id}_proc.gro")
    top_file = os.path.join(run_dir, f"{struct_id}.top")
    boxed_gro = os.path.join(run_dir, f"{struct_id}_boxed.gro")
    tpr_file = os.path.join(run_dir, "em.tpr")
    final_pdb = os.path.join(run_dir, f"{struct_id}.pdb")
    
    # MDP with frozen heavy atoms
    mdp_path = os.path.join(run_dir, "em.mdp")
    with open(mdp_path, "w") as f:
        f.write(
            "integrator      = steep\n"
            "emtol           = 100.0\n"
            "emstep          = 0.01\n"
            "nsteps          = 5000\n"
            "nstlist         = 1\n"
            "cutoff-scheme   = Verlet\n"
            "vdw-type        = Cut-off\n"
            "rvdw            = 1.0\n"
            "rcoulomb        = 1.0\n"
            "pbc             = xyz\n"
            "; Freeze settings\n"
            "freezegrps      = Protein\n"
            "freezedim       = Y Y Y\n"
        )

    try:
        # 2. pdb2gmx
        cmd1 = f"echo 8 | {gmx_path} pdb2gmx -f {input_path} -o {processed_gro} -p {top_file} -water spce -ignh"
        subprocess.run(cmd1, shell=True, check=True, capture_output=True, cwd=run_dir)
        
        # 3. editconf
        cmd2 = f"{gmx_path} editconf -f {processed_gro} -o {boxed_gro} -d 1.0 -c"
        subprocess.run(cmd2, shell=True, check=True, capture_output=True, cwd=run_dir)
        
        # 4. grompp
        cmd3 = f"{gmx_path} grompp -f {mdp_path} -c {boxed_gro} -p {top_file} -o {tpr_file}"
        subprocess.run(cmd3, shell=True, check=True, capture_output=True, cwd=run_dir)
        
        # 5. mdrun
        cmd4 = f"{gmx_path} mdrun -deffnm em"
        subprocess.run(cmd4, shell=True, check=True, capture_output=True, cwd=run_dir)
        
        # 6. Convert back to PDB
        cmd5 = f"echo 'Protein' | {gmx_path} trjconv -s {tpr_file} -f em.gro -o {final_pdb} -pbc mol -ur compact"
        subprocess.run(cmd5, shell=True, check=True, capture_output=True, cwd=run_dir)
        
        print(f"SUCCESS: {struct_id}")

    except subprocess.CalledProcessError as e:
        print(f"FAILED: {struct_id}. Error: {e.stderr.decode()}")


Found 7 structures.
SUCCESS: AEUYGRJBDIISBC-MXQXRXRHNA-O_out_lig1
SUCCESS: AEUYGRJBDIISBC-MXQXRXRHNA-O_out_lig2
SUCCESS: AEUYGRJBDIISBC-MXQXRXRHNA-O_out_lig3
SUCCESS: AEUYGRJBDIISBC-MXQXRXRHNA-O_out_lig4
SUCCESS: APPWDWVMENBZFH-UHFFFAOYNA-N_out_lig1
SUCCESS: APPWDWVMENBZFH-UHFFFAOYNA-N_out_lig2
SUCCESS: APPWDWVMENBZFH-UHFFFAOYNA-N_out_lig3


# Move protein structures from batch run folders

In [8]:

import shutil

base_work_dir = os.path.join(output_root, 'receptor_processed/gromacs/batch_gromacs_runs/')
destination_dir = os.path.join(output_root, 'receptor_processed/gromacs/raw_gromacs_structures/')

os.makedirs(destination_dir, exist_ok=True)
count = 0

print(f"Starting harvest from {base_work_dir}...")

for root, dirs, files in os.walk(base_work_dir):
    for file in files:
        if file.endswith(".pdb"):
            source_path = os.path.join(root, file)
            dest_path = os.path.join(destination_dir, file)
            shutil.copy2(source_path, dest_path)
            count += 1

print(f"Done! Successfully collected {count} relaxed structures into {destination_dir}")


Starting harvest from /home/oliverto/Degree_projectVM/final_pipeline/processed_files/receptor_processed/gromacs/batch_gromacs_runs/...
Done! Successfully collected 7 relaxed structures into /home/oliverto/Degree_projectVM/final_pipeline/processed_files/receptor_processed/gromacs/raw_gromacs_structures/


# Check missing structures 

In [9]:
import os

original_input_dir = os.path.join(output_root, 'receptor_processed/receptor_3_onlyprotein')
batch_runs_dir = os.path.join(output_root, 'receptor_processed/gromacs/raw_gromacs_structures')

if not os.path.exists(original_input_dir):
    print(f"Error: Source directory not found at {original_input_dir}")
    input_ids = set()
else:
    input_ids = set([f.replace(".pdb", "") for f in os.listdir(original_input_dir) if f.endswith('.pdb')])

completed_ids = set()

if os.path.exists(batch_runs_dir):
    contents = os.listdir(batch_runs_dir)
    
    for item in contents:
        # flat directory
        if item.endswith('.pdb'):
            completed_ids.add(item.replace(".pdb", ""))
        
        # subfolder
        else:
            item_path = os.path.join(batch_runs_dir, item)
            if os.path.isdir(item_path):
                expected_pdb = os.path.join(item_path, f"{item}.pdb")
                if os.path.isfile(expected_pdb):
                    completed_ids.add(item)

missing = input_ids - completed_ids

print(f"Pipeline Status:")
print(f"Total Source Structures:    {len(input_ids)}")
print(f"Successfully Processed:      {len(completed_ids)}")
print(f"Missing or Failed:          {len(missing)}")

if len(completed_ids) == 0 and len(contents) > 0:
    print("\nWarning: Destination is not empty, but no valid .pdb structures were recognized.")


Pipeline Status:
Total Source Structures:    7
Successfully Processed:      7
Missing or Failed:          0


# Fix format alterations forced by gromacs 

In [10]:
import os
from Bio.PDB import PDBParser, PDBIO, Select

input_folder = os.path.join(output_root, 'receptor_processed/gromacs/raw_gromacs_structures/')
output_folder = os.path.join(output_root, 'receptor_processed/gromacs/repaired_gromacs_structures/')
target_chain = "B"  # chain ID to assign to all atoms

os.makedirs(output_folder, exist_ok=True)
parser = PDBParser(QUIET=True)
io = PDBIO()

print(f"Repairing PDB columns in {input_folder}...")

for pdb_file in os.listdir(input_folder):
    if not pdb_file.endswith(".pdb"):
        continue
        
    file_path = os.path.join(input_folder, pdb_file)
    structure = parser.get_structure(pdb_file, file_path)
    
    for model in structure:
        for chain in model:
            chain.id = target_chain
            
            for residue in chain:
                for atom in residue:
                    atom_name = atom.get_name().strip()
                    element = ''.join([i for i in atom_name if not i.isdigit()])[0]
                    atom.element = element

    out_path = os.path.join(output_folder, pdb_file)
    io.set_structure(structure)
    io.save(out_path)

print(f"Finished. Fixed PDBs are in: {output_folder}")


Repairing PDB columns in /home/oliverto/Degree_projectVM/final_pipeline/processed_files/receptor_processed/gromacs/raw_gromacs_structures/...
Finished. Fixed PDBs are in: /home/oliverto/Degree_projectVM/final_pipeline/processed_files/receptor_processed/gromacs/repaired_gromacs_structures/


# Identify gromacs-shift and then shift back the atoms before ligand and cofactor reintroduction into the pdb file. 

In [11]:
import os
import re
import numpy as np

def back_shift_relaxed_pdbs(protein_dir, relaxed_dir, output_dir):
    """
    Matches relaxed GROMACS PDBs to original files using '_ligX' naming, 
    calculates the mean shift of all C-alpha atoms, and shifts the relaxed 
    coordinates back.
    """
    if not os.path.exists(output_dir):
        os.makedirs(output_dir)

    relaxed_files = os.listdir(relaxed_dir)
    
    for rel_file in relaxed_files:
        orig_path = os.path.join(protein_dir, rel_file)
        
        if not os.path.exists(orig_path):
            # fallback: search by ID/lig number
            print(f"Direct match not found for {rel_file}, searching by ID/Lig pattern...")
            
            base_id = rel_file.split("_out")[0]
            lig_match = re.search(r'lig(\d+)', rel_file)
            
            if not lig_match:
                print(f"Skipping {rel_file}: No 'lig' number found in name.")
                continue
            
            lig_num = lig_match.group(1)
            orig_file = None
            
            for f in os.listdir(protein_dir):
                if f.startswith(base_id) and f"_lig{lig_num}.pdb" in f:
                    orig_file = f
                    orig_path = os.path.join(protein_dir, orig_file)
                    break
            
            if not orig_file:
                print(f"Warning: Could not find original source for {rel_file} in {protein_dir}")
                continue
        else:
            orig_file = rel_file

        print(f"Processing: {rel_file} <--> {orig_file}")

        shift_vector = calculate_ca_shift(
            orig_path,
            os.path.join(relaxed_dir, rel_file)
        )
    
        if shift_vector is None:
            print(f"Error: No matching C-alpha atoms found for {rel_file}. Skipping.")
            continue

        out_path = os.path.join(output_dir, rel_file)
        apply_inverse_shift(os.path.join(relaxed_dir, rel_file), out_path, shift_vector)
        print(f"Saved refitted PDB to: {out_path}\n")

def parse_ca_coords(pdb_path):
    """Extracts C-alpha coordinates indexed by (Residue Number, Residue Name)."""
    coords = {}
    with open(pdb_path, 'r') as f:
        for line in f:
            if line.startswith(("ATOM", "HETATM")):
                atom_name = line[12:16].strip()
                if atom_name == "CA":
                    res_name = line[17:20].strip()
                    res_num = line[22:26].strip()
                    try:
                        x = float(line[30:38])
                        y = float(line[38:46])
                        z = float(line[46:54])
                        coords[(res_num, res_name)] = np.array([x, y, z])
                    except ValueError:
                        continue
    return coords

def calculate_ca_shift(orig_path, relaxed_path):
    """Calculates the mean (Relaxed - Original) vector for all matching C-alphas."""
    orig_cas = parse_ca_coords(orig_path)
    rel_cas = parse_ca_coords(relaxed_path)
    
    shifts = []
    for key in orig_cas:
        if key in rel_cas:
            shifts.append(rel_cas[key] - orig_cas[key])
    
    if not shifts:
        return None
    
    mean_shift = np.mean(shifts, axis=0)
    std_shift = np.std(shifts, axis=0)
    print(f"Mean shift vector: {mean_shift}, shift stdv: {std_shift}")

    if np.any(std_shift > 0.005): # Slightly increased tolerance for relaxation movement
        print(f"Note: High variance in shift ({std_shift}). Structure has local relaxation changes.")
    
    return mean_shift

def apply_inverse_shift(input_pdb, output_pdb, shift_vector):
    """Subtracts the shift vector from every atom in the PDB."""
    with open(input_pdb, 'r') as fin, open(output_pdb, 'w') as fout:
        for line in fin:
            if line.startswith(("ATOM", "HETATM")):
                try:
                    x = float(line[30:38]) - shift_vector[0]
                    y = float(line[38:46]) - shift_vector[1]
                    z = float(line[46:54]) - shift_vector[2]
                    
                    new_line = line[:30] + f"{x:8.3f}{y:8.3f}{z:8.3f}" + line[54:]
                    fout.write(new_line)
                except ValueError:
                    fout.write(line)
            else:
                fout.write(line)


protein_dir = os.path.join(output_root, 'receptor_processed/receptor_3_onlyprotein/')
relaxed_dir = os.path.join(output_root, 'receptor_processed/gromacs/repaired_gromacs_structures/')
output_dir = os.path.join(output_root, 'receptor_processed/receptor_4_relaxed_reshifted/')

back_shift_relaxed_pdbs(protein_dir, relaxed_dir, output_dir)


Processing: AEUYGRJBDIISBC-MXQXRXRHNA-O_out_lig1.pdb <--> AEUYGRJBDIISBC-MXQXRXRHNA-O_out_lig1.pdb
Mean shift vector: [55.98997683 58.90006564 36.00016602], shift stdv: [0.00295517 0.00297602 0.00302235]
Saved refitted PDB to: /home/oliverto/Degree_projectVM/final_pipeline/processed_files/receptor_processed/receptor_4_relaxed_reshifted/AEUYGRJBDIISBC-MXQXRXRHNA-O_out_lig1.pdb

Processing: AEUYGRJBDIISBC-MXQXRXRHNA-O_out_lig2.pdb <--> AEUYGRJBDIISBC-MXQXRXRHNA-O_out_lig2.pdb
Mean shift vector: [55.98997683 58.90006564 36.01016602], shift stdv: [0.00295517 0.00297602 0.00302235]
Saved refitted PDB to: /home/oliverto/Degree_projectVM/final_pipeline/processed_files/receptor_processed/receptor_4_relaxed_reshifted/AEUYGRJBDIISBC-MXQXRXRHNA-O_out_lig2.pdb

Processing: AEUYGRJBDIISBC-MXQXRXRHNA-O_out_lig3.pdb <--> AEUYGRJBDIISBC-MXQXRXRHNA-O_out_lig3.pdb
Mean shift vector: [55.99997683 58.90006564 36.01016602], shift stdv: [0.00295517 0.00297602 0.00302235]
Saved refitted PDB to: /home/olivert

# Script for parsing a pdb file and renaming residues according to protonation states. E.g. "ASP" into protonation-state specific res names like "ASH", important for PE and NPE parameters.

In [12]:
import os
import MDAnalysis as mda

source_folder = os.path.join(output_root, 'receptor_processed/receptor_4_relaxed_reshifted')
target_folder = os.path.join(output_root, 'receptor_processed/receptor_5_resiude_names')

if not os.path.exists(target_folder):
    os.makedirs(target_folder)

def fix_protonated_residues(input_path, output_path):
    """Rename protonated ASP/GLU/HIS/SER based on sidechain H atoms for MD readiness."""
    u = mda.Universe(input_path)
    
    for res in u.residues:
        # ASP -> ASH (Protonated)
        if res.resname == "ASP":
            if any(a.name == "HD2" for a in res.atoms):
                res.resname = "ASH"
                print(f"   [{res.resid}] ASP -> ASH")
        
        # GLU -> GLH (Protonated)
        elif res.resname == "GLU":
            if any(a.name == "HE2" for a in res.atoms):
                res.resname = "GLH"
                print(f"   [{res.resid}] GLU -> GLH")
        
        # HIS Tautomers/Protonation
        elif res.resname == "HIS":
            has_hd1 = any(a.name == "HD1" for a in res.atoms)
            has_he2 = any(a.name == "HE2" for a in res.atoms)
            
            if has_hd1 and has_he2:
                res.resname = "HIP" # Fully protonated (Charged)
                print(f"   [{res.resid}] HIS -> HIP")
            elif has_hd1:
                res.resname = "HID" # Delta tautomer
                print(f"   [{res.resid}] HIS -> HID")
            elif has_he2:
                res.resname = "HIE" # Epsilon tautomer
                print(f"   [{res.resid}] HIS -> HIE")
        
        # SER N-terminal check
        elif res.resname == "SER":
            has_h2 = any(a.name == "H2" for a in res.atoms)
            has_h3 = any(a.name == "H3" for a in res.atoms)
            if has_h3 and has_h2:
                for atom in res.atoms:
                    if atom.name == "H":
                        atom.name = "H1"
                        print(f"   [{res.resid}] SER: H -> H1")
                        break
    
    u.atoms.write(output_path)

print(f"Running MDAnalysis name fixes on {source_folder}...")

for filename in os.listdir(source_folder):
    if filename.endswith(".pdb"):
        base_name = os.path.splitext(filename)[0]
        output_filename = f"{base_name}.pdb"
        
        full_input_path = os.path.join(source_folder, filename)
        full_output_path = os.path.join(target_folder, output_filename)
        
        try:
            fix_protonated_residues(full_input_path, full_output_path)
            print(f"Done: {output_filename}")
        except Exception as e:
            print(f"Error on {filename}: {e}")

print(f"\nAll protonation states standardized in: {target_folder}")


Running MDAnalysis name fixes on /home/oliverto/Degree_projectVM/final_pipeline/processed_files/receptor_processed/receptor_4_relaxed_reshifted...
   [14] HIS -> HID
   [31] HIS -> HIE
   [40] HIS -> HIE
Done: AEUYGRJBDIISBC-MXQXRXRHNA-O_out_lig1.pdb
   [14] HIS -> HID
   [31] HIS -> HIE
   [40] HIS -> HIE
Done: AEUYGRJBDIISBC-MXQXRXRHNA-O_out_lig2.pdb
   [14] HIS -> HID
   [31] HIS -> HIE
   [40] HIS -> HIE
Done: AEUYGRJBDIISBC-MXQXRXRHNA-O_out_lig3.pdb
   [14] HIS -> HID
   [31] HIS -> HIE
   [40] HIS -> HIE


/home/oliverto/envs/gromacs-env/lib/python3.14/site-packages/MDAnalysis/coordinates/PDB.py:885: UserWarning: Unit cell dimensions not found. CRYST1 record set to unitary values.
  warnings.warn(
/home/oliverto/envs/gromacs-env/lib/python3.14/site-packages/MDAnalysis/coordinates/PDB.py:1282: UserWarning: Found no information for attr: 'formalcharges' Using default value of '0'
  warnings.warn(


Done: AEUYGRJBDIISBC-MXQXRXRHNA-O_out_lig4.pdb
   [14] HIS -> HID
   [31] HIS -> HIE
   [40] HIS -> HIE
Done: APPWDWVMENBZFH-UHFFFAOYNA-N_out_lig1.pdb
   [14] HIS -> HID
   [31] HIS -> HIE
   [40] HIS -> HIE
Done: APPWDWVMENBZFH-UHFFFAOYNA-N_out_lig2.pdb
   [14] HIS -> HID
   [31] HIS -> HIE
   [40] HIS -> HIE
Done: APPWDWVMENBZFH-UHFFFAOYNA-N_out_lig3.pdb

All protonation states standardized in: /home/oliverto/Degree_projectVM/final_pipeline/processed_files/receptor_processed/receptor_5_resiude_names


# Fix fix cofactor after being cut out of reference pdb-file

In [13]:
source_folder = os.path.join(input_root, 'NNMT_SAH/')
target_folder = os.path.join(output_root, 'cofactor_processed')

if not os.path.exists(target_folder):
    os.makedirs(target_folder)

def fix_ligand_pdb_fixed_width(input_path, output_path):
    with open(input_path, 'r') as f:
        lines = f.readlines()

    ligand_atom_no_ticker = 1
    new_lines = []

    for line in lines:
        if line.startswith('ATOM') or line.startswith('HETATM'):
            # Using exact PDB column slicing
            record  = line[0:6]    # ATOM/HETATM
            serial  = line[6:11]   # Atom ID
            name    = line[12:16]  # Atom Name
            resName = line[17:20]  # Residue Name
            chainID = line[21:22]  # Chain
            resSeq  = line[22:26]  # Residue Index
            coords  = line[30:54]  # X, Y, Z
            occ     = line[54:60]  # Occupancy
            temp    = line[60:66]  # Temp factor
            element_col = line[76:78].strip()

            # element logic
            if element_col == 'A':
                element = 'C'
            elif len(element_col) > 1:
                element = element_col[0].upper()
            elif not element_col: # If element col is empty, guess from name
                element = name.strip()[0]
            else:
                element = element_col

            # UNL to LIG
            if resName.strip() == 'UNL':
                resName = 'LIG'
                chainID = 'B'
                resSeq  = f"{ligand_atom_no_ticker:>4}"
                ligand_atom_no_ticker += 1

            formatted_line = (
                f"{record}{serial} {name} {resName} {chainID}{resSeq}    "
                f"{coords}{occ}{temp}          {element:>2s}\n"
            )
            new_lines.append(formatted_line)
        else:
            new_lines.append(line)

    with open(output_path, 'w') as f:
        f.writelines(new_lines)

# Process the directory
for filename in os.listdir(source_folder):
    if filename.endswith(".pdb"):
        base_name = os.path.splitext(filename)[0]
        output_filename = f"{base_name}_p.pdb"
        fix_ligand_pdb_fixed_width(os.path.join(source_folder, filename), 
                                   os.path.join(target_folder, output_filename))
        print(f"Fixed formatting for: {output_filename}")


Fixed formatting for: NNMT_SAH_p.pdb


# script for splitting .sdf files into single conformation .pdb files, whilst simultaneously removing the surrounding residues

In [14]:
import os
from rdkit import Chem

source_folder = os.path.join(input_root, 'ligands')
target_folder = os.path.join(output_root, 'ligand_processed/ligands_0_pdb')

if not os.path.exists(target_folder):
    os.makedirs(target_folder)

def extract_clean_ligand(sdf_path, output_base_path, target_inchi_key):
    """
    Reads an SDF, identifies the ligand by matching the InChIKey 
    of the largest fragments, and saves the matching fragment.
    """
    suppl = Chem.SDMolSupplier(sdf_path, removeHs=False)
    mismatch_count = 0
    
    for i, mol in enumerate(suppl):
        if mol is None:
            continue
        
        frags = Chem.GetMolFrags(mol, asMols=True)
        sorted_frags = sorted(frags, key=lambda x: x.GetNumAtoms(), reverse=True)
        
        match_found = False
        for idx, frag in enumerate(sorted_frags):
            try:
                frag_inchi_key = Chem.MolToInchiKey(frag).split('-')[0]
                if frag_inchi_key == target_inchi_key:
                    if idx > 0:
                        print(f"   ! Mismatch warning: Ligand found in fragment rank {idx+1} (not the largest).")
                        mismatch_count += 1
                    
                    pose_index = i + 1
                    output_filename = f"{output_base_path}_lig{pose_index}.pdb"
                    writer = Chem.PDBWriter(output_filename)
                    writer.write(frag)
                    writer.close()
                    match_found = True
                    print(f"   -> Saved Pose {pose_index}: {os.path.basename(output_filename)} ({frag.GetNumAtoms()} atoms)")
                    break
            except Exception:
                continue
                
        if not match_found:
            return False, mismatch_count
            
    return True, mismatch_count

print(f"Cleaning ligands and converting to PDB in: {source_folder}...")
unmatched_files = []
total_mismatches = 0

for filename in os.listdir(source_folder):
    if filename.endswith(".sdf"):
        base_name = os.path.splitext(filename)[0]
        # Filename format: ABGXADJDTPFFSZ-NMEKOBJZNA-O_out.sdf
        target_inchi = base_name.split('-')[0]
        
        full_input_path = os.path.join(source_folder, filename)
        output_base_path = os.path.join(target_folder, base_name)
        
        print(f"\nProcessing file: {filename}")
        try:
            success, mismatches = extract_clean_ligand(full_input_path, output_base_path, target_inchi)
            total_mismatches += mismatches
            if not success:
                print(f"   ! Could not find matching fragment for {filename}")
                unmatched_files.append(filename)
        except Exception as e:
            print(f"   ! Error processing {filename}: {e}")
            unmatched_files.append(filename)

print("\n" + "="*40)
print("Summary")
print("="*40)
print(f"Total files with fragment mismatches: {total_mismatches}")
if unmatched_files:
    print(f"Files that could not be matched ({len(unmatched_files)}):")
    for f in unmatched_files:
        print(f" - {f}")
else:
    print("All files successfully matched.")


Cleaning ligands and converting to PDB in: /home/oliverto/Degree_projectVM/final_pipeline/raw_files/ligands...

Processing file: AEUYGRJBDIISBC-MXQXRXRHNA-O_out.sdf
   -> Saved Pose 1: AEUYGRJBDIISBC-MXQXRXRHNA-O_out_lig1.pdb (30 atoms)
   -> Saved Pose 2: AEUYGRJBDIISBC-MXQXRXRHNA-O_out_lig2.pdb (30 atoms)
   -> Saved Pose 3: AEUYGRJBDIISBC-MXQXRXRHNA-O_out_lig3.pdb (30 atoms)
   -> Saved Pose 4: AEUYGRJBDIISBC-MXQXRXRHNA-O_out_lig4.pdb (30 atoms)

Processing file: APPWDWVMENBZFH-UHFFFAOYNA-N_out.sdf
   -> Saved Pose 1: APPWDWVMENBZFH-UHFFFAOYNA-N_out_lig1.pdb (22 atoms)
   -> Saved Pose 2: APPWDWVMENBZFH-UHFFFAOYNA-N_out_lig2.pdb (22 atoms)
   -> Saved Pose 3: APPWDWVMENBZFH-UHFFFAOYNA-N_out_lig3.pdb (22 atoms)

Summary
Total files with fragment mismatches: 0
All files successfully matched.


# Ligand-pdb clean-up post conversion, for MDA compatibility. Charge information saved. 

In [15]:
import csv

lig_chain_id = "C"
source_folder = os.path.join(output_root, 'ligand_processed/ligands_0_pdb')
target_folder = os.path.join(output_root, 'ligand_processed/ligands_1_cleanup_n_charge')
charge_data_target_folder = os.path.join(output_root, 'ligand_processed')

if not os.path.exists(target_folder):
    os.makedirs(target_folder)

charge_records = []

def fix_ligand_pdb_fixed_width(input_path, output_path, filename, charge_list):
    with open(input_path, 'r') as f:
        lines = f.readlines()

    fixed_res_seq = f"{1:>4}" 
    new_lines = []

    for line in lines:
        if line.startswith('ATOM') or line.startswith('HETATM'):
            # Slicing the standard PDB columns
            record  = line[0:6]    
            serial  = line[6:11]   
            name    = line[12:16]  
            resName = line[17:20]  
            chainID = line[21:22]  
            coords  = line[30:54]  
            occ     = line[54:60]  
            temp    = line[60:66]  
            
            # Column 76+ often contains the element + formal charge (e.g., "N1+")
            raw_element_section = line[76:].strip()
            element_col = line[76:78].strip()

            # formal charge detection
            if '+' in raw_element_section or '-' in raw_element_section:
                charge_list.append({
                    "Ligand_File": filename,
                    "Atom_Name": name.strip(),
                    "Charge_Key": raw_element_section
                })

            # element parsing
            if element_col == 'A':
                element = 'C'
            elif len(element_col) > 1:
                # If it's "N1", we take "N". If it's "FE", we take "FE".
                element = element_col[0].upper() if element_col[1].isdigit() else element_col.upper()
            elif not element_col: 
                element = name.strip()[0]
            else:
                element = element_col

            if resName.strip() in ['UNL', 'LIG']:
                resName = 'LIG'
                chainID = lig_chain_id
                current_res_seq = fixed_res_seq
            else:
                current_res_seq = line[22:26]

            formatted_line = (
                f"{record}{serial} {name} {resName} {chainID}{current_res_seq}    "
                f"{coords}{occ}{temp}          {element:>2s}\n"
            )
            new_lines.append(formatted_line)
        else:
            new_lines.append(line)

    with open(output_path, 'w') as f:
        f.writelines(new_lines)

# Process the directory
for filename in os.listdir(source_folder):
    if filename.endswith(".pdb"):
        output_filename = f"{os.path.splitext(filename)[0]}.pdb"
        fix_ligand_pdb_fixed_width(
            os.path.join(source_folder, filename), 
            os.path.join(target_folder, output_filename),
            filename,
            charge_records
        )
        print(f"Fixed: {output_filename}")

output_csv = os.path.join(charge_data_target_folder, "ligand_charges.txt")

if charge_records:
    keys = charge_records[0].keys()
    with open(output_csv, 'w', newline='') as f:
        dict_writer = csv.DictWriter(f, fieldnames=keys)
        dict_writer.writeheader()
        dict_writer.writerows(charge_records)
    print(f"\nDone! Saved {len(charge_records)} charge entries to {output_csv}")
else:
    print("\nNo charged atoms found.")


Fixed: AEUYGRJBDIISBC-MXQXRXRHNA-O_out_lig1.pdb
Fixed: AEUYGRJBDIISBC-MXQXRXRHNA-O_out_lig2.pdb
Fixed: AEUYGRJBDIISBC-MXQXRXRHNA-O_out_lig3.pdb
Fixed: AEUYGRJBDIISBC-MXQXRXRHNA-O_out_lig4.pdb
Fixed: APPWDWVMENBZFH-UHFFFAOYNA-N_out_lig1.pdb
Fixed: APPWDWVMENBZFH-UHFFFAOYNA-N_out_lig2.pdb
Fixed: APPWDWVMENBZFH-UHFFFAOYNA-N_out_lig3.pdb

Done! Saved 4 charge entries to /home/oliverto/Degree_projectVM/final_pipeline/processed_files/ligand_processed/ligand_charges.txt


# Final script for joining the processed protein, ligand and cofactor files together into final complete pdb-structure. 

In [16]:
import re 

def shift_conect_line(line, offset):
    parts = [line[i:i+5] for i in range(6, len(line.strip()), 5)]
    shifted_parts = []
    for p in parts:
        if p.strip():
            shifted_val = int(p) + offset
            shifted_parts.append(f"{shifted_val:>5}")
    return "CONECT" + "".join(shifted_parts) + "\n"

def merge_complexes(protein_dir, ligand_dir, sah_path, output_dir):
    if not os.path.exists(output_dir):
        os.makedirs(output_dir)

    sah_atoms = []
    sah_conects = []
    with open(sah_path, 'r') as f:
        for line in f:
            if line.startswith(("ATOM", "HETATM")):
                sah_atoms.append(line)
            elif line.startswith("CONECT"):
                sah_conects.append(line)

    for prot_file in [f for f in os.listdir(protein_dir) if f.endswith(".pdb")]:
        base_id = prot_file.split("_out")[0]
        
        model_match = re.search(r'lig(\d+)', prot_file)
        if not model_match:
            print(f"Skipping {prot_file}: No model number found.")
            continue
            
        model_num = model_match.group(1)

        # match ligand pose by number
        matching_ligands = [f for f in os.listdir(ligand_dir) 
                            if f.startswith(base_id) and f"_lig{model_num}" in f]
        
        if not matching_ligands:
            print(f"Warning: No matching ligand pose found for {prot_file} (searched for lig{model_num})")
            continue

        for lig_file in matching_ligands:
            output_name = f"{os.path.splitext(lig_file)[0]}_complex.pdb"
            out_path = os.path.join(output_dir, output_name)
            
            final_lines = []
            atom_serial = 1

            # A. Process Protein
            with open(os.path.join(protein_dir, prot_file), 'r') as f:
                for line in f:
                    if line.startswith(("ATOM", "HETATM")):
                        new_line = line[:6] + f"{atom_serial:>5}" + line[11:]
                        final_lines.append(new_line)
                        atom_serial += 1
            
            # B. Process SAH
            original_sah_start = int(sah_atoms[0][6:11])
            sah_shift = (atom_serial - original_sah_start)
            
            for line in sah_atoms:
                new_line = line[:6] + f"{atom_serial:>5}" + line[11:]
                final_lines.append(new_line)
                atom_serial += 1
            
            for line in sah_conects:
                final_lines.append(shift_conect_line(line, sah_shift))

            # C. Process Ligand
            lig_atoms = []
            lig_conects = []
            with open(os.path.join(ligand_dir, lig_file), 'r') as f:
                for line in f:
                    if line.startswith(("ATOM", "HETATM")):
                        lig_atoms.append(line)
                    elif line.startswith("CONECT"):
                        lig_conects.append(line)

            original_lig_start = int(lig_atoms[0][6:11])
            lig_shift = (atom_serial - original_lig_start)

            for line in lig_atoms:
                new_line = line[:6] + f"{atom_serial:>5}" + line[11:]
                final_lines.append(new_line)
                atom_serial += 1
            
            for line in lig_conects:
                final_lines.append(shift_conect_line(line, lig_shift))

            final_lines.append("END\n")

            with open(out_path, 'w') as f:
                f.writelines(final_lines)
            
            print(f"Created Matched Complex: {output_name} (using {prot_file})")

LIG_FOLDER = os.path.join(output_root, 'ligand_processed/ligands_1_cleanup_n_charge')
PROT_FOLDER = os.path.join(output_root, 'receptor_processed/receptor_5_resiude_names')
SAH_FILE = os.path.join(output_root, 'cofactor_processed/NNMT_SAH_p.pdb')

OUT_FOLDER = os.path.join(output_root, 'final_complexes')

merge_complexes(PROT_FOLDER, LIG_FOLDER, SAH_FILE, OUT_FOLDER)


Created Matched Complex: AEUYGRJBDIISBC-MXQXRXRHNA-O_out_lig1_complex.pdb (using AEUYGRJBDIISBC-MXQXRXRHNA-O_out_lig1.pdb)
Created Matched Complex: AEUYGRJBDIISBC-MXQXRXRHNA-O_out_lig2_complex.pdb (using AEUYGRJBDIISBC-MXQXRXRHNA-O_out_lig2.pdb)
Created Matched Complex: AEUYGRJBDIISBC-MXQXRXRHNA-O_out_lig3_complex.pdb (using AEUYGRJBDIISBC-MXQXRXRHNA-O_out_lig3.pdb)
Created Matched Complex: AEUYGRJBDIISBC-MXQXRXRHNA-O_out_lig4_complex.pdb (using AEUYGRJBDIISBC-MXQXRXRHNA-O_out_lig4.pdb)
Created Matched Complex: APPWDWVMENBZFH-UHFFFAOYNA-N_out_lig1_complex.pdb (using APPWDWVMENBZFH-UHFFFAOYNA-N_out_lig1.pdb)
Created Matched Complex: APPWDWVMENBZFH-UHFFFAOYNA-N_out_lig2_complex.pdb (using APPWDWVMENBZFH-UHFFFAOYNA-N_out_lig2.pdb)
Created Matched Complex: APPWDWVMENBZFH-UHFFFAOYNA-N_out_lig3_complex.pdb (using APPWDWVMENBZFH-UHFFFAOYNA-N_out_lig3.pdb)


# Function that splits sdf files into single poses (without converting to pdb), for Interaction Fingerprint script 

In [17]:
import os
from rdkit import Chem

source_folder = os.path.join(input_root, 'ligands')
target_folder = os.path.join(output_root, 'ligand_processed/ligands_split_sdf')

os.makedirs(target_folder, exist_ok=True)

def split_sdf_poses_with_verification(sdf_path, output_dir, base_name, target_inchi_key):
    """
    Reads a multi-pose SDF, identifies the fragment matching the InChIKey,
    and saves each pose as a separate clean SDF.
    """
    suppl = Chem.SDMolSupplier(sdf_path, removeHs=False, sanitize=False)
    mismatch_warnings = 0
    match_count = 0
    
    for i, mol in enumerate(suppl):
        if mol is None:
            continue
        
        frags = Chem.GetMolFrags(mol, asMols=True)
        sorted_frags = sorted(frags, key=lambda x: x.GetNumAtoms(), reverse=True)
        
        match_found = False
        for idx, frag in enumerate(sorted_frags):
            try:
                frag_inchi = Chem.MolToInchiKey(frag).split('-')[0]
                
                if frag_inchi == target_inchi_key:
                    if idx > 0:
                        print(f"   ! Mismatch warning: Ligand is fragment rank {idx+1} for pose {i+1}")
                        mismatch_warnings += 1
                    
                    pose_index = i + 1
                    output_filename = os.path.join(output_dir, f"{base_name}_lig{pose_index}.sdf")
                    
                    writer = Chem.SDWriter(output_filename)
                    writer.write(frag)
                    writer.close()
                    
                    print(f"   -> Saved: {os.path.basename(output_filename)} ({frag.GetNumAtoms()} atoms)")
                    match_found = True
                    match_count += 1
                    break
            except Exception as e:
                continue
        
        if not match_found:
            print(f"   ! Warning: No InChIKey match found for Pose {i+1} in {os.path.basename(sdf_path)}")
    
    return match_count, mismatch_warnings

print(f"Splitting and verifying SDFs in: {source_folder}...")

unmatched_files = []
total_mismatches = 0

for filename in os.listdir(source_folder):
    if filename.endswith(".sdf"):
        # Extract InChIKey from filename (e.g., ABGXADJDTPFFSZ-NMEKOBJZNA-O_out.sdf -> ABGXADJDTPFFSZ)
        base_id = os.path.splitext(filename)[0]
        target_inchi = base_id.split('-')[0]
        
        full_input_path = os.path.join(source_folder, filename)
        
        print(f"\nProcessing: {filename}")
        try:
            matches, mismatches = split_sdf_poses_with_verification(full_input_path, target_folder, base_id, target_inchi)
            total_mismatches += mismatches
            if matches == 0:
                unmatched_files.append(filename)
        except Exception as e:
            print(f"   ! Critical error splitting {filename}: {e}")
            unmatched_files.append(filename)

print("\n" + "="*40)
print("Summary")
print("="*40)
print(f"Total poses where ligand was not the largest fragment: {total_mismatches}")
if unmatched_files:
    print(f"Files with ZERO matching poses ({len(unmatched_files)}):")
    for f in unmatched_files:
        print(f" - {f}")
else:
    print("All files contained at least one valid InChIKey match.")

print(f"\nDone. Verified files are in: {target_folder}")


Splitting and verifying SDFs in: /home/oliverto/Degree_projectVM/final_pipeline/raw_files/ligands...

Processing: AEUYGRJBDIISBC-MXQXRXRHNA-O_out.sdf
   -> Saved: AEUYGRJBDIISBC-MXQXRXRHNA-O_out_lig1.sdf (30 atoms)
   -> Saved: AEUYGRJBDIISBC-MXQXRXRHNA-O_out_lig2.sdf (30 atoms)
   -> Saved: AEUYGRJBDIISBC-MXQXRXRHNA-O_out_lig3.sdf (30 atoms)
   -> Saved: AEUYGRJBDIISBC-MXQXRXRHNA-O_out_lig4.sdf (30 atoms)

Processing: APPWDWVMENBZFH-UHFFFAOYNA-N_out.sdf
   -> Saved: APPWDWVMENBZFH-UHFFFAOYNA-N_out_lig1.sdf (22 atoms)
   -> Saved: APPWDWVMENBZFH-UHFFFAOYNA-N_out_lig2.sdf (22 atoms)
   -> Saved: APPWDWVMENBZFH-UHFFFAOYNA-N_out_lig3.sdf (22 atoms)

Summary
Total poses where ligand was not the largest fragment: 0
All files contained at least one valid InChIKey match.

Done. Verified files are in: /home/oliverto/Degree_projectVM/final_pipeline/processed_files/ligand_processed/ligands_split_sdf
